In [ ]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [ ]:
filename = "nate"
directory = "output"

color_image = cv2.imread(f"{directory}/{filename}.color.png")
depth_image = cv2.imread(f"{directory}/{filename}.depth.png", cv2.IMREAD_UNCHANGED)
depth_image = cv2.cvtColor(depth_image, cv2.COLOR_BGR2GRAY)
color_image = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)
# invert depth image
# if depth_image.dtype == np.uint16:
#     depth_image = 65535 - depth_image
# else:
#     depth_image = 255 - depth_image

# depth_image = (depth_image * 0.0001).astype(np.uint16)
# color_raw = o3d.io.read_image(f"{directory}/{filename}.color.png")
color_raw = o3d.geometry.Image(color_image)
depth_raw = o3d.geometry.Image(depth_image)
print(np.max(depth_image), np.min(depth_image))

In [ ]:
def load_rgbd_image(filename: str, directory: str, depth_exp: float = 1.0):
    color_image = cv2.imread(f"{directory}/{filename}.color.png")
    depth_image = cv2.imread(f"{directory}/{filename}.depth.png", cv2.IMREAD_UNCHANGED)
    depth_image = cv2.cvtColor(depth_image, cv2.COLOR_BGR2GRAY)
    if depth_exp != 1.0:
        depth_type = depth_image.dtype
        if depth_type == np.uint16:
            depth_norm = depth_image / 65535.0
        else:
            depth_norm = depth_image / 255.0
        depth_image = np.power(depth_norm, depth_exp)
        if depth_type == np.uint16:
            depth_image = (depth_image * 65535).astype(np.uint16)
        else:
            depth_image = (depth_image * 255).astype(np.uint8)

    color_image = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)
    color_raw = o3d.geometry.Image(color_image)
    depth_raw = o3d.geometry.Image(depth_image)
    rgbd_image = o3d.geometry.RGBDImage.create_from_color_and_depth(
        color_raw, depth_raw, depth_trunc=100000000, convert_rgb_to_intensity=False
    )
    return rgbd_image

In [ ]:
def rgbd_to_points(rgbd_image: o3d.cpu.pybind.geometry.RGBDImage, focal_point_scalar=1.0):
    width, height = np.asarray(rgbd_image.depth).shape
    fx, fy = width * focal_point_scalar, height * focal_point_scalar
    cx, cy = width / 2, height / 2
    intrinsics = o3d.camera.PinholeCameraIntrinsic(width, height, fx, fy, cx, cy)
    point_cloud = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd_image, intrinsics)
    # flip the point cloud so it's right side up
    point_cloud.transform([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])
    return point_cloud

In [ ]:
rgbd_image = load_rgbd_image("nate", "output", 0.6)
pcd = rgbd_to_points(rgbd_image)
o3d.visualization.draw_geometries([pcd])

In [ ]:
width, height = depth_image.shape[1], depth_image.shape[0]
fx, fy = width, height * 2 # Focal length in pixels
cx, cy = width / 2, height / 2  # Center of the image
intrinsics = o3d.camera.PinholeCameraIntrinsic(width, height, fx, fy, cx, cy)
depth_scale = 1000.0   # Adjust as per your depth scale
depth_trunc = 10000000.0

rgbd_image = o3d.geometry.RGBDImage.create_from_color_and_depth(
    color_raw,
    depth_raw,
    depth_scale=depth_scale,  # Adjust this depending on your depth image format
    depth_trunc=depth_trunc,  # Truncate depth values greater than this value
    convert_rgb_to_intensity=False,
)

# Create a point cloud
point_cloud = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd_image, intrinsics)
# point_cloud_downsampled = point_cloud.voxel_down_sample(voxel_size=0.03)
# flip the point cloud
# point_cloud.transform([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])


In [ ]:
o3d.visualization.draw_geometries([point_cloud])

In [ ]:
o3d.visualization.draw_geometries([point_cloud_downsampled])

In [ ]:
point_cloud_downsampled.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
o3d.visualization.draw_geometries([point_cloud_downsampled], point_show_normal=True)

In [ ]:
cl, ind = point_cloud.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
inlier_cloud = point_cloud.select_by_index(ind)
o3d.visualization.draw_geometries([inlier_cloud])

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import zscore

len(point_cloud.points)

pcl_points = np.asarray(point_cloud.points)
depth_values = depth_image.flatten() / 65535
depth_values = 1 - depth_values
depth_values = zscore(depth_values)
pcl_points = zscore(pcl_points)


num_points = 10000


test_pcd = pcl_points[5502053:5542453]
test_depth = depth_values[5502053:5542453]

print(test_pcd)
print(test_depth)

index = list(range(len(test_pcd)))
# test_depth = [[]]
plt.plot(index, test_pcd[:, 2])
plt.plot(index, test_depth)
plt.show()


# plt.scatter(test_pcd[:, 0], test_pcd[:, 1], c=test_depth, cmap='viridis')
# plt.colorbar()
# plt.show()

In [ ]:

def rgbd_to_points(color_image: np.ndarray, depth_image: np.ndarray, xy_scale: float = 0.001):
    assert color_image.shape[:2] == depth_image.shape[:2]
    points = []
    colors = []
    max_val = 255
    if depth_image.dtype == np.uint16:
        max_val = 65535

    depth_values = depth_image.astype(np.float32) / max_val
    depth_values = 1 - depth_values

    # Create a grid of coordinates
    x = np.linspace(0, depth_values.shape[0], depth_values.shape[0], endpoint=False, dtype=np.float32)
    y = np.linspace(0, depth_values.shape[1], depth_values.shape[1], endpoint=False, dtype=np.float32)
    x *= xy_scale
    y *= xy_scale
    xx, yy = np.meshgrid(x, y, indexing='ij')
    zz = depth_values

    # Stack the coordinates
    points = np.stack((xx, yy, zz), axis=-1).reshape(-1, 3)

    # Normalize and reshape color image
    colors = (color_image / 255).reshape(-1, 3)


    manual_pcd = o3d.geometry.PointCloud()

    manual_pcd.points = o3d.utility.Vector3dVector(points)
    manual_pcd.colors = o3d.utility.Vector3dVector(colors)
    return manual_pcd, points, colors

In [ ]:
point_cloud, points, colors = rgbd_to_points(color_image, depth_image)
o3d.visualization.draw_geometries([point_cloud])

In [ ]:
print(points[1])

In [ ]:
manual_pcd = o3d.geometry.PointCloud()

points = []
colors = []
xy_scale = 0.001
depth_values = depth_image / 65535
depth_values = 1 - depth_values

for x in range(depth_values.shape[0]):
    for y in range(depth_values.shape[1]):
        z = depth_values[x, y]
        points.append([x * xy_scale, y * xy_scale, z])
        color = color_image[x, y] / 255
        colors.append(color)


manual_pcd.points = o3d.utility.Vector3dVector(points)
manual_pcd.colors = o3d.utility.Vector3dVector(colors)
o3d.visualization.draw_geometries([manual_pcd])

In [ ]:
point_cloud_downsampled.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))

cl, ind = point_cloud_downsampled.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)

valid_points = point_cloud_downsampled.select_by_index(ind)

with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        valid_points, depth=9)
print(mesh)
o3d.visualization.draw_geometries([mesh])

In [ ]:
print('visualize densities')
densities = np.asarray(densities)
density_colors = plt.get_cmap('plasma')(
    (densities - densities.min()) / (densities.max() - densities.min()))
density_colors = density_colors[:, :3]
density_mesh = o3d.geometry.TriangleMesh()
density_mesh.vertices = mesh.vertices
density_mesh.triangles = mesh.triangles
density_mesh.triangle_normals = mesh.triangle_normals
density_mesh.vertex_colors = o3d.utility.Vector3dVector(density_colors)
o3d.visualization.draw_geometries([density_mesh],
                                  zoom=0.664,
                                  front=[-0.4761, -0.4698, -0.7434],
                                  lookat=[1.8900, 3.2596, 0.9284],
                                  up=[0.2304, -0.8825, 0.4101])

In [ ]:
print('remove low density vertices')
vertices_to_remove = densities < np.quantile(densities, 0.1)
mesh.remove_vertices_by_mask(vertices_to_remove)
print(mesh)
o3d.visualization.draw_geometries([mesh],
                                  zoom=0.664,
                                  front=[-0.4761, -0.4698, -0.7434],
                                  lookat=[1.8900, 3.2596, 0.9284],
                                  up=[0.2304, -0.8825, 0.4101])

In [ ]:
N = 2000

# point_cloud_sampled = point_cloud.sample_points_poisson_disk(N)

# fit to unit cube
point_cloud.scale(
    1 / np.max(point_cloud.get_max_bound() - point_cloud.get_min_bound()),
    center=point_cloud.get_center(),
)
point_cloud.colors = o3d.utility.Vector3dVector(np.random.uniform(0, 1, size=(N, 3)))
# o3d.visualization.draw_geometries([point_cloud])


print('voxelization')
voxel_grid = o3d.geometry.VoxelGrid.create_from_point_cloud(point_cloud,
                                                            voxel_size=0.01)
o3d.visualization.draw_geometries([voxel_grid])

In [ ]:
def display_inlier_outlier(cloud, ind):
    inlier_cloud = cloud.select_by_index(ind)
    outlier_cloud = cloud.select_by_index(ind, invert=True)

    print("Showing outliers (red) and inliers (gray): ")
    outlier_cloud.paint_uniform_color([1, 0, 0])
    # inlier_cloud.paint_uniform_color([0.8, 0.8, 0.8])
    o3d.visualization.draw_geometries(
        [inlier_cloud, outlier_cloud],
        zoom=0.3412,
        front=[0.4257, -0.2125, -0.8795],
        lookat=[2.6172, 2.0475, 1.532],
        up=[-0.0694, -0.9768, 0.2024],
    )

In [ ]:
cl, ind = point_cloud.remove_statistical_outlier(nb_neighbors=20,
                                                    std_ratio=2.0)
display_inlier_outlier(point_cloud, ind)

In [ ]:
print("Radius oulier removal")
cl, ind = downpcd.remove_radius_outlier(nb_points=16, radius=0.05)
display_inlier_outlier(point_cloud, ind)

In [ ]:
valid_points = point_cloud.select_by_index(ind)
o3d.visualization.draw_geometries([valid_points])

In [ ]:
valid_points.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))

print('run Poisson surface reconstruction')
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        valid_points, depth=9)
print(mesh)
o3d.visualization.draw_geometries([mesh],
                                  zoom=0.664,
                                  front=[-0.4761, -0.4698, -0.7434],
                                  lookat=[1.8900, 3.2596, 0.9284],
                                  up=[0.2304, -0.8825, 0.4101])


In [ ]:
print(dir(valid_points))

In [ ]:
point_cloud.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))

# Optionally orient the normals
point_cloud.orient_normals_towards_camera_location(camera_location=[0, 0, 0])

# Apply the Ball Pivoting algorithm
radii = [0.005, 0.01, 0.02, 0.04]
mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
       point_cloud, o3d.utility.DoubleVector(radii))

# Visualize the mesh
o3d.visualization.draw_geometries([mesh])

In [ ]:
o3d.io.write_point_cloud("output_point_cloud.ply", point_cloud)

In [ ]:
np.asarray(rgbd_image.depth).max()

In [ ]:
plt.subplot(1, 2, 1)
plt.title('grayscale image')
plt.imshow(rgbd_image.color)
plt.subplot(1, 2, 2)
plt.title('depth image')
plt.imshow(rgbd_image.depth)
plt.show()

In [ ]:
pcd = o3d.geometry.PointCloud.create_from_rgbd_image(
    rgbd_image,
    o3d.camera.PinholeCameraIntrinsic(
        o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault
    ),
)
# Flip it, otherwise the pointcloud will be upside down
# pcd.transform([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])
# o3d.visualization.draw_geometries([pcd], zoom=0.5)

In [ ]:
# pcd.transform([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])
o3d.visualization.draw_geometries([pcd])